# 04. Базовый анализ подготовленных данных

## Тема

**Загрузка и интеграция данных из различных форматов. Инструменты для сбора данных. Основы Python для обработки данных**

В предыдущем ноутбуке мы собрали единую таблицу:

```text
sales_prepared.csv
```

В ней объединены:

- продажи;
- товары;
- регионы;
- клиенты;
- план продаж.

Теперь научимся выполнять первичный аналитический разбор:

- выбирать столбцы;
- фильтровать строки;
- сортировать данные;
- создавать расчетные поля;
- использовать `groupby`;
- использовать `agg`;
- строить `pivot_table`;
- применять `describe`;
- считать `corr`;
- сохранять аналитический результат.

## 1. Цель ноутбука

После выполнения ноутбука вы должны уметь:

1. Загружать подготовленную аналитическую таблицу.
2. Выбирать нужные столбцы.
3. Фильтровать строки по условиям.
4. Сортировать таблицу.
5. Создавать расчетный показатель `revenue`.
6. Группировать данные через `groupby`.
7. Использовать несколько агрегатов через `agg`.
8. Создавать сводные таблицы через `pivot_table`.
9. Использовать `describe()` для первичной статистики.
10. Использовать `corr()` для поиска связи между числовыми показателями.
11. Сохранять результаты анализа в CSV и Excel.

## 2. Импорт библиотек и поиск подготовленного файла

Нам нужен файл:

```text
data/prepared/sales_prepared.csv
```

Если файл не найден, значит нужно сначала выполнить ноутбук:

```text
03_data_integration.ipynb
```

In [ ]:
import pandas as pd
from pathlib import Path

print("pandas:", pd.__version__)

In [ ]:
def find_prepared_file() -> Path:
    """Найти файл sales_prepared.csv в типовых местах."""
    current_dir = Path.cwd()

    candidates = [
        current_dir / "data" / "prepared" / "sales_prepared.csv",
        current_dir.parent / "data" / "prepared" / "sales_prepared.csv",
        current_dir.parent.parent / "data" / "prepared" / "sales_prepared.csv",
        current_dir / "sales_prepared.csv",
        current_dir.parent / "sales_prepared.csv",
        Path("/mnt/data/data/prepared/sales_prepared.csv"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return current_dir / "data" / "prepared" / "sales_prepared.csv"


prepared_path = find_prepared_file()

print("Ожидаемый файл:")
print(prepared_path)

if not prepared_path.exists():
    raise FileNotFoundError(
        "Файл sales_prepared.csv не найден. "
        "Сначала выполните 03_data_integration.ipynb, чтобы создать подготовленную таблицу."
    )

## 3. Загрузка `sales_prepared.csv`

Загрузим подготовленную таблицу.

In [ ]:
df = pd.read_csv(prepared_path)

print("Файл загружен:", prepared_path)
print("Размер таблицы:", df.shape)

df.head()

## 4. Первичный осмотр таблицы

Перед анализом снова проверяем:

- размер;
- список столбцов;
- типы данных;
- пропуски.

In [ ]:
print("Размер таблицы:")
print(df.shape)

print("\nСписок столбцов:")
print(df.columns.tolist())

print("\nИнформация о таблице:")
df.info()

In [ ]:
print("Пропуски по столбцам:")
df.isna().sum()

## 5. Приведение дат после загрузки CSV

Даже если в предыдущем ноутбуке дата была типом `datetime`, после сохранения в CSV и повторной загрузки она снова может стать текстом.

Преобразуем дату заказа обратно в datetime.

In [ ]:
if "order_date" in df.columns:
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

if "registration_date" in df.columns:
    df["registration_date"] = pd.to_datetime(df["registration_date"], errors="coerce")

print(df[["order_date"]].head())
print("\nТип order_date:", df["order_date"].dtype)

# Часть 1. Выбор столбцов

## 6. Выбор одного столбца

Если выбрать один столбец, pandas вернет `Series`.

In [ ]:
df["net_revenue"].head()

## 7. Выбор нескольких столбцов

Если выбрать список столбцов, pandas вернет `DataFrame`.

In [ ]:
main_columns = [
    "sale_id",
    "order_date",
    "region_name",
    "category",
    "channel",
    "quantity",
    "unit_price",
    "net_revenue",
    "gross_profit",
]

df[main_columns].head()

## 8. Создание аналитической витрины

Создадим компактную таблицу только с теми полями, которые нужны для первичного анализа.

In [ ]:
analysis_df = df[main_columns].copy()

analysis_df.head()

# Часть 2. Фильтрация строк

## 9. Фильтрация по одному условию

Выберем только завершенные заказы, если в таблице есть поле `order_status`.

In [ ]:
if "order_status" in df.columns:
    completed_orders = df[df["order_status"] == "completed"]
else:
    completed_orders = df.copy()

print("Все строки:", df.shape[0])
print("Завершенные заказы:", completed_orders.shape[0])

completed_orders.head()

## 10. Фильтрация по числовому условию

Выберем продажи, где выручка после скидки больше 50 000.

In [ ]:
high_revenue_orders = df[df["net_revenue"] > 50_000]

print("Количество заказов с net_revenue > 50 000:", high_revenue_orders.shape[0])

high_revenue_orders[[
    "sale_id",
    "product_name",
    "category",
    "region_name",
    "net_revenue",
]].head()

## 11. Фильтрация по нескольким условиям

Выберем завершенные онлайн-заказы с положительной выручкой.

Важно: каждое условие нужно заключать в скобки.

In [ ]:
if "order_status" in df.columns:
    online_completed = df[
        (df["order_status"] == "completed") &
        (df["channel"] == "online") &
        (df["net_revenue"] > 0)
    ]
else:
    online_completed = df[
        (df["channel"] == "online") &
        (df["net_revenue"] > 0)
    ]

print("Количество строк:", online_completed.shape[0])

online_completed.head()

## 12. Фильтрация через `isin()`

Выберем продажи только по нескольким каналам.

In [ ]:
selected_channels = ["online", "marketplace"]

channel_filtered = df[df["channel"].isin(selected_channels)]

print("Каналы:", selected_channels)
print("Количество строк:", channel_filtered.shape[0])

channel_filtered[["sale_id", "channel", "net_revenue"]].head()

# Часть 3. Сортировка

## 13. Сортировка по одному столбцу

Найдем самые крупные продажи по `net_revenue`.

In [ ]:
top_sales = df.sort_values("net_revenue", ascending=False)

top_sales[[
    "sale_id",
    "product_name",
    "category",
    "region_name",
    "net_revenue",
]].head(10)

## 14. Сортировка по нескольким столбцам

Отсортируем данные по региону и выручке.

In [ ]:
df.sort_values(
    by=["region_name", "net_revenue"],
    ascending=[True, False]
)[[
    "sale_id",
    "region_name",
    "product_name",
    "net_revenue",
]].head(15)

# Часть 4. Создание расчетных показателей

## 15. Создание `revenue`

В таблице уже есть несколько расчетных полей:

- `gross_revenue`;
- `net_revenue`;
- `gross_profit`.

Но для учебной практики создадим новый столбец `revenue`.

Под `revenue` будем понимать выручку после скидки:

```text
revenue = quantity * unit_price * (1 - discount_percent / 100)
```

In [ ]:
df["revenue"] = df["quantity"] * df["unit_price"] * (1 - df["discount_percent"] / 100)

df[[
    "sale_id",
    "quantity",
    "unit_price",
    "discount_percent",
    "revenue",
    "net_revenue",
]].head(10)

## 16. Проверка нового показателя

Сравним `revenue` и `net_revenue`.

Если формулы одинаковые, разница должна быть близка к нулю.  
Но если в данных есть пропуски или ошибки, могут появиться отличия.

In [ ]:
df["revenue_diff"] = df["revenue"] - df["net_revenue"]

df[["sale_id", "revenue", "net_revenue", "revenue_diff"]].head(10)

In [ ]:
print("Максимальное абсолютное отличие:")
print(df["revenue_diff"].abs().max())

# Часть 5. `describe()` — первичная статистика

## 17. Статистика по числовым столбцам

`describe()` показывает:

- количество непустых значений;
- среднее;
- стандартное отклонение;
- минимум;
- квартили;
- максимум.

In [ ]:
numeric_columns = [
    "quantity",
    "unit_price",
    "discount_percent",
    "gross_revenue",
    "net_revenue",
    "gross_profit",
    "revenue",
]

available_numeric_columns = [col for col in numeric_columns if col in df.columns]

df[available_numeric_columns].describe()

## 18. Статистика по категориальным столбцам

Для текстовых столбцов можно использовать:

```python
describe(include="object")
```

In [ ]:
df.describe(include="object")

# Часть 6. `groupby`

## 19. Выручка по категориям

`groupby()` группирует строки по выбранному признаку.

Пример:

> сгруппировать продажи по категории и посчитать сумму выручки.

In [ ]:
category_revenue = (
    df
    .groupby("category", dropna=False)["revenue"]
    .sum()
    .sort_values(ascending=False)
)

category_revenue

## 20. Выручка по регионам

In [ ]:
region_revenue = (
    df
    .groupby("region_name", dropna=False)["revenue"]
    .sum()
    .sort_values(ascending=False)
)

region_revenue

## 21. Количество заказов по каналам

Посчитаем количество уникальных продаж по каналам.

In [ ]:
channel_orders = (
    df
    .groupby("channel", dropna=False)["sale_id"]
    .nunique()
    .sort_values(ascending=False)
)

channel_orders

# Часть 7. `agg` — несколько показателей сразу

## 22. Агрегация по категориям

Метод `agg()` позволяет посчитать несколько показателей одновременно.

In [ ]:
category_summary = (
    df
    .groupby("category", dropna=False)
    .agg(
        orders_count=("sale_id", "nunique"),
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        avg_revenue=("revenue", "mean"),
        total_profit=("gross_profit", "sum"),
        avg_discount=("discount_percent", "mean"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

category_summary

## 23. Агрегация по регионам и каналам

Теперь сгруппируем по двум признакам:

- регион;
- канал продаж.

In [ ]:
region_channel_summary = (
    df
    .groupby(["region_name", "channel"], dropna=False)
    .agg(
        orders_count=("sale_id", "nunique"),
        total_revenue=("revenue", "sum"),
        total_profit=("gross_profit", "sum"),
        avg_order_revenue=("revenue", "mean"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

region_channel_summary.head(20)

## 24. Средний чек

Средний чек можно считать по-разному.  
В учебном варианте используем:

```text
средний чек = сумма выручки / количество заказов
```

In [ ]:
channel_summary = (
    df
    .groupby("channel", dropna=False)
    .agg(
        orders_count=("sale_id", "nunique"),
        total_revenue=("revenue", "sum"),
        total_profit=("gross_profit", "sum"),
    )
    .reset_index()
)

channel_summary["avg_check"] = (
    channel_summary["total_revenue"] / channel_summary["orders_count"]
)

channel_summary.sort_values("total_revenue", ascending=False)

# Часть 8. `pivot_table`

## 25. Сводная таблица: регионы × категории

`pivot_table()` помогает построить таблицу, похожую на сводную таблицу в Excel.

Создадим сводную таблицу:

- строки — регионы;
- столбцы — категории;
- значения — сумма выручки.

In [ ]:
revenue_pivot = pd.pivot_table(
    df,
    values="revenue",
    index="region_name",
    columns="category",
    aggfunc="sum",
    fill_value=0,
)

revenue_pivot

## 26. Сводная таблица: каналы × категории

Посмотрим, какие категории лучше продаются в разных каналах.

In [ ]:
channel_category_pivot = pd.pivot_table(
    df,
    values="revenue",
    index="channel",
    columns="category",
    aggfunc="sum",
    fill_value=0,
)

channel_category_pivot

## 27. Сводная таблица с несколькими показателями

Можно считать не только сумму, но и количество заказов.

In [ ]:
multi_pivot = pd.pivot_table(
    df,
    values=["revenue", "sale_id"],
    index="region_name",
    columns="channel",
    aggfunc={
        "revenue": "sum",
        "sale_id": "nunique",
    },
    fill_value=0,
)

multi_pivot

# Часть 9. `corr()` — корреляция

## 28. Что такое корреляция

Корреляция показывает, насколько два числовых показателя связаны между собой.

Значения:

| Корреляция | Интерпретация |
|---:|---|
| около `1` | сильная положительная связь |
| около `0` | явной линейной связи нет |
| около `-1` | сильная отрицательная связь |

Важно:

> корреляция не доказывает причинно-следственную связь.

In [ ]:
corr_columns = [
    "quantity",
    "unit_price",
    "discount_percent",
    "gross_revenue",
    "net_revenue",
    "gross_profit",
    "revenue",
]

available_corr_columns = [col for col in corr_columns if col in df.columns]

correlation_matrix = df[available_corr_columns].corr(numeric_only=True)

correlation_matrix

## 29. Корреляция между отдельными показателями

Посмотрим связь между скидкой и выручкой.

In [ ]:
df[["discount_percent", "revenue"]].corr()

### Важное ограничение

Если корреляция между скидкой и выручкой положительная, это не значит, что скидка всегда увеличивает выручку.

Возможно, большие скидки давали на дорогие товары.  
Чтобы сделать вывод, нужен дополнительный анализ.

# Часть 10. Примеры аналитических выводов

## 30. Топ категорий по выручке

In [ ]:
top_categories = category_summary.head(5)

top_categories

Пример вывода:

> Самая высокая выручка приходится на категории из верхних строк таблицы. Эти категории стоит отдельно анализировать по прибыли и среднему чеку.

## 31. Топ регионов по выручке

In [ ]:
top_regions = (
    region_revenue
    .reset_index()
    .rename(columns={"revenue": "total_revenue"})
    .head(5)
)

top_regions

Пример вывода:

> Основная выручка концентрируется в нескольких регионах. Для остальных регионов стоит проверить план продаж и каналы.

## 32. Каналы продаж

In [ ]:
channel_summary.sort_values("total_revenue", ascending=False)

Пример вывода:

> Канал с максимальной выручкой не всегда является каналом с максимальным средним чеком. Поэтому полезно смотреть сразу несколько показателей.

# Часть 11. Сохранение аналитических результатов

## 33. Сохранение отдельных CSV-файлов

Сохраним ключевые аналитические таблицы.

In [ ]:
output_dir = Path("data/output")
output_dir.mkdir(parents=True, exist_ok=True)

category_summary_path = output_dir / "category_summary.csv"
region_channel_summary_path = output_dir / "region_channel_summary.csv"
channel_summary_path = output_dir / "channel_summary.csv"

category_summary.to_csv(category_summary_path, index=False, encoding="utf-8")
region_channel_summary.to_csv(region_channel_summary_path, index=False, encoding="utf-8")
channel_summary.to_csv(channel_summary_path, index=False, encoding="utf-8")

print("Сохранены файлы:")
print(category_summary_path)
print(region_channel_summary_path)
print(channel_summary_path)

## 34. Сохранение Excel-отчета с несколькими листами

Сохраним несколько результатов в один Excel-файл.

In [ ]:
analysis_report_path = output_dir / "basic_analysis_report.xlsx"

with pd.ExcelWriter(analysis_report_path, engine="openpyxl") as writer:
    category_summary.to_excel(writer, sheet_name="category_summary", index=False)
    region_channel_summary.to_excel(writer, sheet_name="region_channel", index=False)
    channel_summary.to_excel(writer, sheet_name="channel_summary", index=False)
    revenue_pivot.to_excel(writer, sheet_name="revenue_pivot")
    channel_category_pivot.to_excel(writer, sheet_name="channel_category")
    correlation_matrix.to_excel(writer, sheet_name="correlation")

print("Excel-отчет сохранен:")
print(analysis_report_path)

print("\nПроверка:")
print(analysis_report_path.exists())

# Часть 12. Мини-задания

Выполните задания самостоятельно.

## Задание 1

Выберите столбцы:

```text
sale_id, order_date, product_name, category, region_name, revenue
```

Выведите первые 10 строк.

In [ ]:
# Ваш код здесь

## Задание 2

Отфильтруйте строки, где `revenue` больше 100 000.

In [ ]:
# Ваш код здесь

## Задание 3

Отсортируйте таблицу по `gross_profit` по убыванию.

Выведите топ-10 строк.

In [ ]:
# Ваш код здесь

## Задание 4

Посчитайте суммарную выручку по `client_type`.

In [ ]:
# Ваш код здесь

## Задание 5

С помощью `agg()` посчитайте по регионам:

- количество заказов;
- сумму выручки;
- среднюю выручку;
- сумму прибыли.

In [ ]:
# Ваш код здесь

## Задание 6

Постройте `pivot_table`:

- строки — `category`;
- столбцы — `channel`;
- значения — сумма `revenue`.

In [ ]:
# Ваш код здесь

## Задание 7

Посчитайте корреляцию между:

```text
quantity, unit_price, discount_percent, revenue
```

In [ ]:
# Ваш код здесь

## Задание 8

Сохраните результат задания 5 в файл:

```text
data/output/region_summary_task.xlsx
```

In [ ]:
# Ваш код здесь

# Часть 13. Контрольные вопросы

Ответьте своими словами:

1. Чем выбор одного столбца отличается от выбора нескольких столбцов?
2. Как отфильтровать строки по одному условию?
3. Как отфильтровать строки по нескольким условиям?
4. Для чего используется `sort_values()`?
5. Зачем создавать расчетный столбец `revenue`?
6. Что делает `describe()`?
7. Что делает `groupby()`?
8. Чем `agg()` удобнее простого `.sum()`?
9. Для чего нужна `pivot_table()`?
10. Что показывает `corr()`?
11. Почему корреляция не доказывает причинность?
12. Зачем сохранять аналитический результат в Excel?

# 35. Итог ноутбука

В этом ноутбуке мы научились:

- загружать подготовленную таблицу `sales_prepared.csv`;
- выбирать столбцы;
- фильтровать строки;
- сортировать данные;
- создавать показатель `revenue`;
- смотреть первичную статистику через `describe()`;
- группировать данные через `groupby()`;
- считать несколько метрик через `agg()`;
- строить сводные таблицы через `pivot_table()`;
- считать корреляции через `corr()`;
- сохранять результаты анализа в CSV и Excel.

Следующий шаг:

```text
05_basic_visualization.ipynb
```

Там мы будем строить графики по результатам анализа.